<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR &mdash; End-to-End Validation</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Run this after NB00 to NB08. One pass over instruments, portfolios, holdings, transactions, market data, valuations, lifecycle events and reconciliation, with a single PASS/FAIL summary.</div>
</div>

<sub>Run last &middot; after IBOR pack NB00 &nbsp;&rarr;&nbsp; NB08</sub>

# MEGA VALIDATION — v8

Run this **after NB00–NB08** have all been run against `ibor-training-v8`.
It verifies instruments, portfolios, holdings, transactions, market data, valuations (grouped by name + SubHoldingKey), lifecycle events, and reconciliation in one pass, and prints a PASS/FAIL summary.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp

with open("secrets.json") as f: secrets = json.load(f)
pat = secrets["api"]["accessToken"]
lusid_factory = lu.extensions.SyncApiClientFactory(
    config_loaders=[lu.extensions.ArgsConfigurationLoader(
        api_url=secrets["api"]["lusidUrl"], access_token=pat)])
lumi = lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))

tp_api = lusid_factory.build(lu.TransactionPortfoliosApi)
instruments_api = lusid_factory.build(lu.InstrumentsApi)
portfolios_api = lusid_factory.build(lu.PortfoliosApi)
valuation_api = lusid_factory.build(lu.AggregationApi)

SCOPE = 'ibor-training-v8'
QUOTE_SCOPE = 'ibor-training-v8-quotes'
EFFECTIVE_AT = "2024-09-30T00:00:00+00:00"

passed = failed = warnings = 0
results = []

def check(name, condition, value, detail=""):
    global passed, failed
    icon = "PASS" if condition else "FAIL"
    if condition: passed += 1
    else: failed += 1
    results.append((icon, name, value, detail))
    color = "\033[92m" if condition else "\033[91m"
    print(f"  {color}{icon}\033[0m  {name}: {value}  {detail}")

def warn(name, value, detail=""):
    global warnings
    warnings += 1
    results.append(("WARN", name, value, detail))
    print(f"  \033[93mWARN\033[0m  {name}: {value}  {detail}")

print("MEGA VALIDATION (v8) STARTING...")
print(f"Scope: {SCOPE}  |  Quote scope: {QUOTE_SCOPE}  |  Date: {EFFECTIVE_AT[:10]}")
print("=" * 80)

In [ ]:
print("\n[1/8] INSTRUMENTS")
print("-" * 40)

try:
    q = f"SELECT COUNT(*) as Cnt FROM Lusid.Instrument WHERE Scope = '{SCOPE}' AND Type = 'Equity'"
    df = lumi.run(q, quiet=True)
    cnt = int(df.iloc[0]['Cnt'])
    check("Total equities", cnt >= 100, f"{cnt} instruments", "Expected >= 100")
except Exception as e:
    check("Total equities", False, "ERROR", str(e)[:100])

# OTC + ZCB must be SimpleInstrument (not native)
for ci, name in [("IBOR-AAPL","Apple"),("IBOR-MSFT","Microsoft"),("IBOR-NVDA","NVIDIA"),
                 ("IBOR-IRS-5Y-SOFR","5Y IRS"),("IBOR-TD-6M-USD","Term Deposit"),
                 ("IBOR-ZCB-2028","Zero Coupon Bond")]:
    try:
        resp = instruments_api.get_instrument(identifier_type="ClientInternal", identifier=ci, scope=SCOPE)
        luid = resp.lusid_instrument_id
        unresolved = (luid == "LUID_ZZZZZZZZ")
        itype = getattr(resp.instrument_definition, "instrument_type", "?") if hasattr(resp,"instrument_definition") else "?"
        # For OTC/ZCB confirm SimpleInstrument
        if ci in ("IBOR-IRS-5Y-SOFR","IBOR-TD-6M-USD","IBOR-ZCB-2028"):
            is_simple = (itype == "SimpleInstrument")
            check(f"{name} is SimpleInstrument", is_simple and not unresolved, f"{luid} ({itype})",
                  "" if is_simple else "EXPECTED SimpleInstrument")
        else:
            check(f"{name} ({ci})", not unresolved, luid, "UNRESOLVED!" if unresolved else "")
    except Exception as e:
        check(f"{name} ({ci})", False, "NOT FOUND", str(e)[:80])

# Bonds + SimpleInstruments + futures/options
try:
    q = f"SELECT COUNT(*) as Cnt FROM Lusid.Instrument WHERE Scope = '{SCOPE}' AND Type IN ('Bond','SimpleInstrument','Future','EquityOption')"
    df = lumi.run(q, quiet=True)
    cnt = int(df.iloc[0]['Cnt'])
    check("Bonds + OTC + F&O instruments", cnt >= 30, f"{cnt} instruments", "Expected >= 30")
except Exception as e:
    check("Bonds + derivatives", False, "ERROR", str(e)[:100])

In [ ]:
print("\n[2/8] PORTFOLIOS")
print("-" * 40)

expected_pfs = ["IBOR-EQ","IBOR-FI","IBOR-MA","IBOR-CASH","IBOR-SP500","IBOR-AITECH","IBOR-BLKC","IBOR-GAGG"]
for pf in expected_pfs:
    try:
        resp = portfolios_api.get_portfolio(scope=SCOPE, code=pf)
        check(f"{pf}", True, f"{resp.display_name}", f"base={resp.base_currency}")
    except Exception as e:
        check(f"{pf}", False, "NOT FOUND", str(e)[:60])

# Sub-holding keys are exposed on the transaction-portfolio DETAILS, not the base portfolio.
for pf in ["IBOR-EQ", "IBOR-MA"]:
    try:
        det = tp_api.get_details(scope=SCOPE, code=pf)
        shks = det.sub_holding_keys or []
        check(f"{pf} sub-holding keys", len(shks) >= 1, f"{len(shks)} SHK(s)",
              ", ".join(k.split('/')[-1] for k in shks))
    except Exception as e:
        # get_details signature varies; treat as non-fatal warning rather than fail
        warn(f"{pf} sub-holding keys", "could not read", str(e)[:70])


In [ ]:
print("\n[3/8] HOLDINGS (at 2024-09-30)")
print("-" * 40)

expected_holdings = {
    "IBOR-EQ": (8,"equities + cash"), "IBOR-FI": (7,"bonds + cash"),
    "IBOR-MA": (10,"multi asset"), "IBOR-SP500": (25,"S&P 500"),
    "IBOR-AITECH": (20,"AI/tech"), "IBOR-BLKC": (20,"blockchain"), "IBOR-GAGG": (20,"global bonds"),
}
for pf,(threshold,desc) in expected_holdings.items():
    try:
        resp = tp_api.get_holdings(scope=SCOPE, code=pf, effective_at=EFFECTIVE_AT)
        positions = [h for h in resp.values if h.holding_type == 'P']
        cash = [h for h in resp.values if h.holding_type == 'B']
        check(f"{pf} positions", len(positions) >= threshold,
              f"{len(positions)} positions, {len(cash)} cash", desc)
    except Exception as e:
        check(f"{pf} positions", False, "ERROR", str(e)[:100])

In [ ]:
print("\n[4/8] TRANSACTIONS")
print("-" * 40)

for pf, min_txns in [("IBOR-EQ",10),("IBOR-FI",7),("IBOR-MA",10),
                     ("IBOR-SP500",25),("IBOR-AITECH",20),("IBOR-BLKC",20),("IBOR-GAGG",20)]:
    try:
        resp = tp_api.get_transactions(scope=SCOPE, code=pf)
        check(f"{pf} transactions", len(resp.values) >= min_txns,
              f"{len(resp.values)} transactions", f"Expected >= {min_txns}")
    except Exception as e:
        check(f"{pf} transactions", False, "ERROR", str(e)[:100])

In [ ]:
print("\n[5/8] MARKET DATA")
print("-" * 40)

def quote_count(idtype, instid):
    q = (f"SELECT COUNT(*) as Cnt FROM Lusid.Instrument.Quote WHERE QuoteScope = '{QUOTE_SCOPE}' "
         f"AND InstrumentIdType = '{idtype}' AND InstrumentId = '{instid}'")
    return int(lumi.run(q, quiet=True).iloc[0]['Cnt'])

try: check("AAPL equity quotes", quote_count("ClientInternal","IBOR-AAPL") >= 10, f"{quote_count('ClientInternal','IBOR-AAPL')} quotes")
except Exception as e: check("AAPL equity quotes", False, "ERROR", str(e)[:80])

try:
    q = f"SELECT Value FROM Lusid.Instrument.Quote WHERE QuoteScope = '{QUOTE_SCOPE}' AND InstrumentId = 'IBOR-UST-4.5-2029' LIMIT 1"
    val = float(lumi.run(q, quiet=True).iloc[0]['Value'])
    check("Bond prices are decimal", val < 2.0, f"{val:.4f}", "Should be <2 (price per 1), not per 100")
except Exception as e: check("Bond prices are decimal", False, "ERROR", str(e)[:80])

# OTC + ZCB MTM quotes present (these drive SimpleInstrument pricing)
for ci in ["IBOR-IRS-5Y-SOFR","IBOR-TD-6M-USD","IBOR-ZCB-2028"]:
    try: check(f"{ci} MTM quotes", quote_count("ClientInternal",ci) >= 5, f"{quote_count('ClientInternal',ci)} quotes")
    except Exception as e: check(f"{ci} MTM quotes", False, "ERROR", str(e)[:80])

# FX both formats
try:
    slash = quote_count("CurrencyPair","TWD/USD"); dot = quote_count("CurrencyPair","TWD.USD")
    # Slash quotes + the recipe Fx.*.* rule are what drive resolution; dot is belt-and-braces.
    check("FX TWD slash quotes present", slash > 0, f"slash={slash}")
    if dot == 0:
        warn("FX TWD dot format", "slash=%d dot=0" % slash,
             "dot-format CurrencyPair not stored; OK if valuations resolve via Fx.*.* rule")
    else:
        check("FX TWD dot format present", True, f"dot={dot}")
except Exception as e: check("FX TWD quotes", False, "ERROR", str(e)[:80])

In [ ]:
print("\n[6/8] VALUATION (grouped by name + SubHoldingKey)")
print("-" * 40)

def value_pf(pf):
    result = valuation_api.get_valuation(
        valuation_request=lm.ValuationRequest(
            recipe_id=lm.ResourceId(scope=SCOPE, code="IBOR-VALUATION-RECIPE"),
            metrics=[lm.AggregateSpec(key="Holding/default/PV", op="Value")],
            group_by=["Instrument/default/Name", "Holding/default/SubHoldingKey"],
            portfolio_entity_ids=[lm.PortfolioEntityId(scope=SCOPE, code=pf, portfolio_entity_type="SinglePortfolio")],
            valuation_schedule=lm.ValuationSchedule(effective_at=EFFECTIVE_AT)))
    lines = 0; total = 0.0
    for row in result.data:
        pv = row.get("Holding/default/PV")
        if pv is not None:
            lines += 1; total += pv
    return lines, total

expected = {
    "IBOR-EQ": (10e6, 14e6), "IBOR-FI": (25e6, 33e6), "IBOR-MA": (18e6, 26e6),
    "IBOR-SP500": (42e6, 58e6), "IBOR-AITECH": (8e6, 13e6),
    "IBOR-BLKC": (3e6, 7e6), "IBOR-GAGG": (115e6, 145e6),
}
for pf,(lo,hi) in expected.items():
    try:
        lines, total = value_pf(pf)
        in_range = lo <= total <= hi
        check(f"{pf} valuation", in_range, f"${total/1e6:.1f}M ({lines} lines)",
              "" if in_range else f"Expected ${lo/1e6:.0f}-{hi/1e6:.0f}M")
    except Exception as e:
        check(f"{pf} valuation", False, "ERROR", str(e)[:110])

In [ ]:
print("\n[7/8] LIFECYCLE EVENTS")
print("-" * 40)

# AAPL 2:1 split: holdings should increase across 2024-08-01
try:
    aapl_luid = instruments_api.get_instrument(identifier_type="ClientInternal", identifier="IBOR-AAPL", scope=SCOPE).lusid_instrument_id
    before = tp_api.get_holdings(scope=SCOPE, code="IBOR-EQ", effective_at="2024-07-31T00:00:00+00:00")
    after  = tp_api.get_holdings(scope=SCOPE, code="IBOR-EQ", effective_at="2024-08-02T00:00:00+00:00")
    b = sum(h.units for h in before.values if h.instrument_uid == aapl_luid)
    a = sum(h.units for h in after.values if h.instrument_uid == aapl_luid)
    check("AAPL stock split (2:1)", a > b, f"Before: {b:,.0f}, After: {a:,.0f}")
except Exception as e:
    check("AAPL stock split", False, "ERROR", str(e)[:100])

# F&O lifecycle events booked in IBOR-MA (NB05)
try:
    resp = tp_api.get_transactions(scope=SCOPE, code="IBOR-MA")
    exercises = [t for t in resp.values if t.type == "CashSettledOptionExercise"]
    expiries  = [t for t in resp.values if t.type == "Expiry"]
    fut_settle= [t for t in resp.values if t.type == "FutureCashSettlement"]
    check("F&O cash-settled exercises (ITM)", len(exercises) >= 4, f"{len(exercises)} events", "AAPL/AMZN/GOOGL/NVDA")
    check("F&O expiries (OTM worthless)", len(expiries) >= 2, f"{len(expiries)} events", "MSFT/META")
    check("Future cash settlements", len(fut_settle) >= 4, f"{len(fut_settle)} events")
except Exception as e:
    check("F&O lifecycle events", False, "ERROR", str(e)[:100])

In [ ]:
print("\n[8/8] RECONCILIATION")
print("-" * 40)

for code_ in ["IBOR-FI-FLAT","CUSTODIAN-MIRROR"]:
    try:
        resp = portfolios_api.get_portfolio(scope=SCOPE, code=code_)
        check(f"{code_} portfolio", True, resp.display_name)
    except Exception as e:
        check(f"{code_} portfolio", False, "NOT FOUND (run NB07)", str(e)[:60])

try:
    recon_api = lusid_factory.build(lu.ReconciliationsApi)
    result = recon_api.reconcile_holdings(
        portfolios_reconciliation_request=lm.PortfoliosReconciliationRequest(
            left=lm.PortfolioReconciliationRequest(
                portfolio_id=lm.ResourceId(scope=SCOPE, code="IBOR-FI-FLAT"), effective_at="2024-09-30"),
            right=lm.PortfolioReconciliationRequest(
                portfolio_id=lm.ResourceId(scope=SCOPE, code="CUSTODIAN-MIRROR"), effective_at="2024-09-30"),
            instrument_property_keys=["Instrument/default/Name"]))
    breaks = [b for b in result.values if abs(b.difference_units) > 0.01]
    check("Recon runs successfully", True, f"{len(result.values)} items, {len(breaks)} breaks")
except Exception as e:
    check("Recon runs", False, "ERROR", str(e)[:100])

In [ ]:
print()
print("=" * 80)
print("  MEGA VALIDATION (v8) COMPLETE")
print(f"  Scope: {SCOPE}")
print("=" * 80)
total = passed + failed + warnings
print(f"  PASSED:   {passed}")
print(f"  FAILED:   {failed}")
print(f"  WARNINGS: {warnings}")
print()
if failed == 0:
    print("  ALL CHECKS PASSED - IBOR Training Pack v8 is READY")
else:
    print("  FAILURES:")
    for icon,name,value,detail in results:
        if icon == "FAIL":
            print(f"    - {name}: {value} {detail}")